# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rebha-ds/flyrank-first-ml-assignment/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.
>
> Completed using Gemini

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...):  ········


In [3]:
import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"

In [7]:
fcqd = con.sql(
    f"""
    CREATE OR REPLACE TABLE cleaned_fact_content_daily_performance AS 
    SELECT DISTINCT
        report_date, 
        client_hash_id, 
        content_hash_id, 
        gsc_impressions, 
        gsc_clicks, 
        gsc_sum_position, 
        ga4_pageviews, 
        ga4_sessions, 
        ga4_users, 
        ga4_engaged_sessions, 
        ga4_total_engagement_sec, 
        sessions_organic, 
        sessions_direct, 
        sessions_referral, 
        sessions_social, 
        sessions_paid, 
        sessions_ai, 
        ai_chatgpt, 
        ai_perplexity, 
        ai_gemini, 
        ai_copilot, 
        ai_claude, 
        ai_meta, 
        ai_other, 
        scroll_events
    FROM read_parquet([
        '{rel}/fact_content_daily_performance/month%3D2026-04/data_0.parquet',
        '{rel}/fact_content_daily_performance/month%3D2026-05/data_0.parquet',
        '{rel}/fact_content_daily_performance/month%3D2026-06/data_0.parquet'
    ], union_by_name=true)
    WHERE report_date >= '2026-04-02' 
      AND ga4_data_available IS NOT NULL
"""
)

In [9]:
feature_frame_may = con.sql(
    f"""
    WITH engaged_sessions_30d AS (
        SELECT 
            content_hash_id,
            SUM(ga4_engaged_sessions) AS engaged_sessions_prev30
        FROM cleaned_fact_content_daily_performance
        WHERE report_date >= DATE '2026-05-01' AND report_date < '2026-06-01'
        GROUP BY content_hash_id
    ),
    daily_metrics_may AS (
        SELECT 
            content_hash_id,
            MEDIAN(gsc_clicks) AS median_clicks_may,
            MEDIAN(gsc_impressions) AS median_impressions_may,
            MEDIAN(ga4_engaged_sessions) AS median_engaged_sessions_may,
            MEDIAN(gsc_sum_position) AS median_sum_position
        FROM cleaned_fact_content_daily_performance
        WHERE report_date >= DATE '2026-05-02' AND report_date < '2026-06-01'
        GROUP BY content_hash_id
    )
    SELECT 
        p.content_hash_id,
        p.median_clicks_may,
        p.median_impressions_may,
        p.median_engaged_sessions_may,
        c.main_intent,
        c.content_type,
        q.impressions_prev30 / 30.0 AS imp_prev30_avg,  -- we don't have many numbers to find median from so, we are simply dividing to find average, we'll use median later on.
        q.avg_position_prev30,
        q.clicks_prev30 / 30.0 AS clicks_prev30_avg,
        e.engaged_sessions_prev30 / 30.0 AS engaged_sessions_prev30_avg,
        p.median_sum_position,
        DATEDIFF('day', c.content_updated_date, DATE '2026-05-31') AS content_updated_days
    FROM daily_metrics_may p
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON p.content_hash_id = c.content_hash_id 
    JOIN read_parquet('{rel}/fact_content_query_90d.parquet') q
        ON p.content_hash_id = q.content_hash_id    
    LEFT JOIN engaged_sessions_30d e
        ON p.content_hash_id = e.content_hash_id
    WHERE c.content_updated_date >= '2026-05-01' AND c.content_updated_date <= '2026-05-31'
      AND c.main_intent IS NOT NULL
      AND c.is_published IS TRUE
"""
).df()

feature_frame_may.head()

,content_hash_id,median_clicks_may,median_impressions_may,median_engaged_sessions_may,main_intent,content_type,imp_prev30_avg,avg_position_prev30,clicks_prev30_avg,engaged_sessions_prev30_avg,median_sum_position,content_updated_days
0,content_ed152ef1018cb619,0.0,6.0,0.0,transactional,keyword article,0.300000,3.888889,0.0,0.0,109.0,11
1,content_d62e1b540bd93971,0.0,13.0,0.0,informational,keyword article,0.000000,NaN,0.0,0.0,245.5,11
2,content_e86ef0f0940bbfe4,0.0,14.5,0.0,informational,keyword article,0.100000,15.000000,0.0,0.0,167.0,13
3,content_b99b97cf47563fd9,0.0,30.5,0.0,informational,keyword article,0.266667,13.125000,0.0,0.0,295.5,11
4,content_5f54d8e4bf2cd0fd,0.0,2.5,0.0,informational,keyword article,0.000000,NaN,0.0,0.0,34.5,11


In [8]:
import lightgbm as lgb
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# 1. Prepare data (Encode categorical columns)
cat_cols = ["main_intent", "content_type"]
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=-1
)
feature_frame_may[cat_cols] = encoder.fit_transform(
    feature_frame_may[cat_cols]
)

# Drop non-feature columns like the hash ID
X_cols = [
    c
    for c in feature_frame_may.columns
    if c not in ["content_hash_id", "avg_position_prev30"]
]

# 2. Split into train (not null) and predict (null) sets
train_df = feature_frame_may[feature_frame_may["avg_position_prev30"].notnull()]
predict_df = feature_frame_may[feature_frame_may["avg_position_prev30"].isnull()]

X_train, y_train = train_df[X_cols], train_df["avg_position_prev30"]
X_predict = predict_df[X_cols]

# 3. Train a LightGBM Regressor (handles millions of rows very quickly)
model = lgb.LGBMRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Fill missing values with predictions
feature_frame_may.loc[
    feature_frame_may["avg_position_prev30"].isnull(), "avg_position_prev30"
] = model.predict(X_predict)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026588 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 885
[LightGBM] [Info] Number of data points in the train set: 490200, number of used features: 10
[LightGBM] [Info] Start training from score 25.424558


In [46]:
feature_frame_may.info()

<class 'pandas.DataFrame'>
RangeIndex: 567655 entries, 0 to 567654
Data columns (total 12 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   content_hash_id              567655 non-null  str    
 1   median_clicks_may            567655 non-null  float64
 2   median_impressions_may       567655 non-null  float64
 3   median_engaged_sessions_may  567655 non-null  float64
 4   main_intent                  567655 non-null  float64
 5   content_type                 567655 non-null  float64
 6   imp_prev30_avg               567655 non-null  float64
 7   avg_position_prev30          567655 non-null  float64
 8   clicks_prev30_avg            567655 non-null  float64
 9   engaged_sessions_prev30_avg  567655 non-null  float64
 10  median_sum_position          567655 non-null  float64
 11  content_updated_days         567655 non-null  int64  
dtypes: float64(10), int64(1), str(1)
memory usage: 65.0 MB


In [60]:
con.sql(f"""
SUMMARIZE
    WITH engaged_sessions_30d AS (
        SELECT 
            content_hash_id,
            SUM(ga4_engaged_sessions) AS engaged_sessions_prev30
        FROM cleaned_fact_content_daily_performance
        WHERE report_date >= DATE '2026-05-01' AND report_date < '2026-06-01'
        GROUP BY content_hash_id
    ),
    daily_metrics_may AS (
        SELECT 
            content_hash_id,
            MEDIAN(gsc_clicks) AS median_clicks_may,
            MEDIAN(gsc_impressions) AS median_impressions_may,
            MEDIAN(ga4_engaged_sessions) AS median_engaged_sessions_may,
            MEDIAN(gsc_sum_position) AS median_sum_position
        FROM cleaned_fact_content_daily_performance
        WHERE report_date >= DATE '2026-05-02' AND report_date < '2026-06-01'
        GROUP BY content_hash_id
    )
    SELECT 
        p.content_hash_id,
        p.median_clicks_may,
        p.median_impressions_may,
        p.median_engaged_sessions_may,
        c.main_intent,
        c.content_type,
        q.impressions_prev30 / 30.0 AS imp_prev30_avg,  -- we don't have many numbers to find median from so, we are simply dividing to find average, we'll use median later on.
        q.avg_position_prev30,
        q.clicks_prev30 / 30.0 AS clicks_prev30_avg,
        e.engaged_sessions_prev30 / 30.0 AS engaged_sessions_prev30_avg,
        p.median_sum_position,
        DATEDIFF('day', c.content_updated_date, DATE '2026-05-31') AS content_updated_days
    FROM daily_metrics_may p
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON p.content_hash_id = c.content_hash_id 
    JOIN read_parquet('{rel}/fact_content_query_90d.parquet') q
        ON p.content_hash_id = q.content_hash_id    
    LEFT JOIN engaged_sessions_30d e
        ON p.content_hash_id = e.content_hash_id
    WHERE c.content_updated_date >= '2026-05-01' AND c.content_updated_date <= '2026-05-31'
      AND c.main_intent IS NOT NULL
      AND c.is_published IS TRUE
"""
).df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,content_hash_id,VARCHAR,content_000005d4ced12088,content_fffff09da8a25da6,36614,NaN,NaN,NaN,NaN,NaN,567655,0.00
1,median_clicks_may,DOUBLE,0.0,37.0,37,0.8007460517391726,2.837357566364037,0.0,0.0,0.06447651270867376,567655,0.00
2,median_impressions_may,DOUBLE,0.0,4963.5,1217,241.22169099188767,483.7622293146482,19.840653961558907,64.72499467003584,223.97562808815985,567655,0.00
3,median_engaged_sessions_may,DOUBLE,0.0,3.0,6,0.01018047934044446,0.12028189341491725,0.0,0.0,0.0,567655,0.00
4,main_intent,VARCHAR,commercial,transactional,4,NaN,NaN,NaN,NaN,NaN,567655,0.00
5,content_type,VARCHAR,comparison article,keyword article,2,NaN,NaN,NaN,NaN,NaN,567655,0.00
6,imp_prev30_avg,DOUBLE,0.0,3897.7,1572,0.8247050291696593,11.275179532905199,0.08320321630713615,0.22659197462375874,0.503319916312191,567655,0.00
7,avg_position_prev30,DOUBLE,0.0,672.0,76992,25.42455754244788,26.120629036375167,6.532695269202808,11.574720949754443,41.49571433578881,567655,13.64
8,clicks_prev30_avg,DOUBLE,0.0,11.9,59,0.0019541211945048537,0.02949795359404897,0.0,0.0,0.0,567655,0.00
9,engaged_sessions_prev30_avg,DOUBLE,0.0,3.2333333333333334,34,0.04730972157383742,0.1539231990432542,0.0,0.0,0.03333333333333333,567655,0.00


In [24]:
print(feature_frame_may.shape)

(567655, 12)


In [12]:
import pandas as pd

# List of your original raw columns
raw_cols = [
    'content_updated_days',
    'median_impressions_may',
    'median_clicks_may',
    'median_engaged_sessions_may'
]

for col in raw_cols:
    print(f"=== Raw Distribution for: {col} ===")
    
    # q=4 creates 4 bins where each bin contains roughly 25% of the data points
    # retbins=True lets us see the actual threshold values for each quartile
    binned, bin_edges = pd.qcut(feature_frame_may[col], q=4, retbins=True, duplicates='drop')
    
    print(binned.value_counts(sort=False))
    print(f"Bin Edges (Thresholds): {bin_edges}\n")
categorical_cols = ['main_intent', 'content_type']

cat_cols = ["main_intent", "content_type"]

for i, col in enumerate(cat_cols):
    print(f"=== Distribution for: {col} ===")
    
    # Get counts and percentages
    counts = feature_frame_may[col].value_counts(dropna=False)
    percentages = feature_frame_may[col].value_counts(normalize=True, dropna=False) * 100
    
    summary_df = pd.DataFrame({
        'Count': counts, 
        'Percentage (%)': percentages.round(2)
    })
    
    # Map the numeric indices back to original category names from the encoder
    categories = encoder.categories_[i]
    index_mapping = {idx: name for idx, name in enumerate(categories)}
    index_mapping[-1.0] = 'Unknown / Out-of-Vocabulary'
    index_mapping[-1] = 'Unknown / Out-of-Vocabulary'
    
    # Apply the mapping to the index
    summary_df.index = summary_df.index.map(index_mapping).fillna('Missing (NaN)')
    
    print(summary_df)
    print("\n" + "-"*40 + "\n")

=== Raw Distribution for: content_updated_days ===
content_updated_days
(1.999, 11.0]    455604
(11.0, 25.0]     112051
Name: count, dtype: int64
Bin Edges (Thresholds): [ 2. 11. 25.]

=== Raw Distribution for: median_impressions_may ===
median_impressions_may
(-0.001, 20.0]     144645
(20.0, 64.5]       139470
(64.5, 224.5]      141709
(224.5, 4963.5]    141831
Name: count, dtype: int64
Bin Edges (Thresholds): [   0.    20.    64.5  224.5 4963.5]

=== Raw Distribution for: median_clicks_may ===
median_clicks_may
(-0.001, 37.0]    567655
Name: count, dtype: int64
Bin Edges (Thresholds): [ 0. 37.]

=== Raw Distribution for: median_engaged_sessions_may ===
median_engaged_sessions_may
(-0.001, 3.0]    567655
Name: count, dtype: int64
Bin Edges (Thresholds): [0. 3.]

=== Distribution for: main_intent ===
                Count  Percentage (%)
main_intent                          
informational  383821           67.62
transactional   98606           17.37
commercial      83293           14.6

<b>We need to remove `content_type` since we don't have enough categories as compared to original data.</b>

In [11]:
feature_frame_may = con.sql(
    f"""
    WITH engaged_sessions_30d AS (
        SELECT 
            content_hash_id,
            SUM(ga4_engaged_sessions) AS engaged_sessions_prev30
        FROM cleaned_fact_content_daily_performance
        WHERE report_date >= DATE '2026-05-01' AND report_date < '2026-06-01'
        GROUP BY content_hash_id
    ),
    daily_metrics_may AS (
        SELECT 
            content_hash_id,
            MEDIAN(gsc_clicks) AS median_clicks_may,
            MEDIAN(gsc_impressions) AS median_impressions_may,
            MEDIAN(ga4_engaged_sessions) AS median_engaged_sessions_may,
            MEDIAN(gsc_sum_position) / NULLIF(MEDIAN(gsc_impressions), 0) AS median_avg_position_may
        FROM cleaned_fact_content_daily_performance
        WHERE report_date >= DATE '2026-05-02' AND report_date < '2026-06-01'
        GROUP BY content_hash_id
    )
    SELECT 
        p.content_hash_id,
        p.median_clicks_may,
        p.median_impressions_may,
        p.median_engaged_sessions_may,
        c.main_intent,
        q.impressions_prev30 / 30.0 AS imp_prev30_avg,  -- we don't have many numbers to find median from so, we are simply dividing to find average, we'll use median later on.
        q.avg_position_prev30,
        q.clicks_prev30 / 30.0 AS clicks_prev30_avg,
        e.engaged_sessions_prev30 / 30.0 AS engaged_sessions_prev30_avg,
        p.median_avg_position_may,
        DATEDIFF('day', c.content_updated_date, DATE '2026-05-31') AS content_updated_days
    FROM daily_metrics_may p
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON p.content_hash_id = c.content_hash_id 
    JOIN read_parquet('{rel}/fact_content_query_90d.parquet') q
        ON p.content_hash_id = q.content_hash_id    
    LEFT JOIN engaged_sessions_30d e
        ON p.content_hash_id = e.content_hash_id
    WHERE c.content_updated_date >= '2026-05-01' AND c.content_updated_date <= '2026-05-31'
      AND c.main_intent IS NOT NULL
      AND c.is_published IS TRUE
"""
).df()

feature_frame_may.head()

,content_hash_id,median_clicks_may,median_impressions_may,median_engaged_sessions_may,main_intent,imp_prev30_avg,avg_position_prev30,clicks_prev30_avg,engaged_sessions_prev30_avg,median_avg_position_may,content_updated_days
0,content_14a7a440e8873d91,0.0,7.0,0.0,informational,0.500000,34.200000,0.0,0.0,25.285714,13
1,content_4589b35a7e12c8e4,0.0,7.0,0.0,informational,0.200000,65.500000,0.0,0.0,6.857143,11
2,content_5e6182004a3e3cdb,0.0,3.0,0.0,commercial,0.100000,69.333333,0.0,0.0,71.500000,13
3,content_5ca1b43f9a4d0b01,0.0,3.5,0.0,informational,1.700000,70.470588,0.0,0.0,68.142857,13
4,content_8ed57b4607088cc1,0.0,46.5,0.0,commercial,0.533333,45.625000,0.0,0.0,72.645161,13


In [13]:
import lightgbm as lgb
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# 1. Prepare data (Encode categorical columns)
cat_cols = ["main_intent"]
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=-1
)
feature_frame_may[cat_cols] = encoder.fit_transform(
    feature_frame_may[cat_cols]
)

# Drop non-feature columns like the hash ID
X_cols = [
    c
    for c in feature_frame_may.columns
    if c not in ["content_hash_id", "avg_position_prev30"]
]

# 2. Split into train (not null) and predict (null) sets
train_df = feature_frame_may[feature_frame_may["avg_position_prev30"].notnull()]
predict_df = feature_frame_may[feature_frame_may["avg_position_prev30"].isnull()]

X_train, y_train = train_df[X_cols], train_df["avg_position_prev30"]
X_predict = predict_df[X_cols]

# 3. Train a LightGBM Regressor (handles millions of rows very quickly)
model = lgb.LGBMRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Fill missing values with predictions
feature_frame_may.loc[
    feature_frame_may["avg_position_prev30"].isnull(), "avg_position_prev30"
] = model.predict(X_predict)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023314 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 883
[LightGBM] [Info] Number of data points in the train set: 490200, number of used features: 9
[LightGBM] [Info] Start training from score 25.424558


In [14]:
feature_frame_may.info()

<class 'pandas.DataFrame'>
RangeIndex: 567655 entries, 0 to 567654
Data columns (total 11 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   content_hash_id              567655 non-null  str    
 1   median_clicks_may            567655 non-null  float64
 2   median_impressions_may       567655 non-null  float64
 3   median_engaged_sessions_may  567655 non-null  float64
 4   main_intent                  567655 non-null  float64
 5   imp_prev30_avg               567655 non-null  float64
 6   avg_position_prev30          567655 non-null  float64
 7   clicks_prev30_avg            567655 non-null  float64
 8   engaged_sessions_prev30_avg  567655 non-null  float64
 9   median_avg_position_may      560844 non-null  float64
 10  content_updated_days         567655 non-null  int64  
dtypes: float64(9), int64(1), str(1)
memory usage: 60.6 MB


In [68]:
con.sql(f"""
SUMMARIZE
    WITH engaged_sessions_30d AS (
        SELECT 
            content_hash_id,
            SUM(ga4_engaged_sessions) AS engaged_sessions_prev30
        FROM cleaned_fact_content_daily_performance
        WHERE report_date >= DATE '2026-05-01' AND report_date < '2026-06-01'
        GROUP BY content_hash_id
    ),
    daily_metrics_may AS (
        SELECT 
            content_hash_id,
            MEDIAN(gsc_clicks) AS median_clicks_may,
            MEDIAN(gsc_impressions) AS median_impressions_may,
            MEDIAN(ga4_engaged_sessions) AS median_engaged_sessions_may,
            MEDIAN(gsc_sum_position) / NULLIF(MEDIAN(gsc_impressions), 0) AS median_sum_position_approx
        FROM cleaned_fact_content_daily_performance
        WHERE report_date >= DATE '2026-05-02' AND report_date < '2026-06-01'
        GROUP BY content_hash_id
    )
    SELECT 
        p.content_hash_id,
        p.median_clicks_may,
        p.median_impressions_may,
        p.median_engaged_sessions_may,
        c.main_intent,
        q.impressions_prev30 / 30.0 AS imp_prev30_avg,  -- we don't have many numbers to find median from so, we are simply dividing to find average, we'll use median later on.
        q.avg_position_prev30,
        q.clicks_prev30 / 30.0 AS clicks_prev30_avg,
        e.engaged_sessions_prev30 / 30.0 AS engaged_sessions_prev30_avg,
        p.median_sum_position_approx,
        DATEDIFF('day', c.content_updated_date, DATE '2026-05-31') AS content_updated_days
    FROM daily_metrics_may p
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON p.content_hash_id = c.content_hash_id 
    JOIN read_parquet('{rel}/fact_content_query_90d.parquet') q
        ON p.content_hash_id = q.content_hash_id    
    LEFT JOIN engaged_sessions_30d e
        ON p.content_hash_id = e.content_hash_id
    WHERE c.content_updated_date >= '2026-05-01' AND c.content_updated_date <= '2026-05-31'
      AND c.main_intent IS NOT NULL
      AND c.is_published IS TRUE
"""
).df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,content_hash_id,VARCHAR,content_000005d4ced12088,content_fffff09da8a25da6,36614,NaN,NaN,NaN,NaN,NaN,567655,0.00
1,median_clicks_may,DOUBLE,0.0,37.0,37,0.8007460517391726,2.8373575663640276,0.0,0.0,0.131947794606169,567655,0.00
2,median_impressions_may,DOUBLE,0.0,4963.5,1217,241.22169099188767,483.76222931464804,19.782328072679945,64.52477544397517,223.31063268707976,567655,0.00
3,median_engaged_sessions_may,DOUBLE,0.0,3.0,6,0.01018047934044446,0.12028189341491732,0.0,0.0,0.0,567655,0.00
4,main_intent,VARCHAR,commercial,transactional,4,NaN,NaN,NaN,NaN,NaN,567655,0.00
5,imp_prev30_avg,DOUBLE,0.0,3897.7,1572,0.8247050291696648,11.275179532905245,0.08500929707371363,0.22708786497327282,0.5030502033801438,567655,0.00
6,avg_position_prev30,DOUBLE,0.0,672.0,76992,25.42455754244784,26.120629036375178,6.521405602559831,11.582226746272875,41.477865673226376,567655,13.64
7,clicks_prev30_avg,DOUBLE,0.0,11.9,59,0.0019541211945048533,0.029497953594048998,0.0,0.0,0.0,567655,0.00
8,engaged_sessions_prev30_avg,DOUBLE,0.0,3.2333333333333334,34,0.04730972157383653,0.1539231990432542,0.0,0.0,0.03333333333333333,567655,0.00
9,median_sum_position_approx,DOUBLE,0.0,105.16666666666667,23397,17.812838947478944,15.42109508224862,7.055195899461782,11.107150258738203,24.45735437033485,567655,1.20


In [70]:
print(feature_frame_may.shape)

(567655, 11)


In [74]:
import pandas as pd

# List of your original raw columns
raw_cols = [
    'content_updated_days',
    'median_impressions_may',
    'median_clicks_may',
    'median_engaged_sessions_may',
    'median_avg_position_may',
    'imp_prev30_avg',
    'avg_position_prev30',
    'clicks_prev30_avg',
    'engaged_sessions_prev30_avg'
]

for col in raw_cols:
    print(f"=== Raw Distribution for: {col} ===")
    
    # q=4 creates 4 bins where each bin contains roughly 25% of the data points
    # retbins=True lets us see the actual threshold values for each quartile
    binned, bin_edges = pd.qcut(feature_frame_may[col], q=4, retbins=True, duplicates='drop')
    
    print(binned.value_counts(sort=False))
    print(f"Bin Edges (Thresholds): {bin_edges}\n")


for i, col in enumerate(cat_cols):
    print(f"=== Distribution for: {col} ===")
    
    # Get counts and percentages
    counts = feature_frame_may[col].value_counts(dropna=False)
    percentages = feature_frame_may[col].value_counts(normalize=True, dropna=False) * 100
    
    summary_df = pd.DataFrame({
        'Count': counts, 
        'Percentage (%)': percentages.round(2)
    })
    
    # Map the numeric indices back to original category names from the encoder
    categories = encoder.categories_[i]
    index_mapping = {idx: name for idx, name in enumerate(categories)}
    index_mapping[-1.0] = 'Unknown / Out-of-Vocabulary'
    index_mapping[-1] = 'Unknown / Out-of-Vocabulary'
    
    # Apply the mapping to the index
    summary_df.index = summary_df.index.map(index_mapping).fillna('Missing (NaN)')
    
    print(summary_df)
    print("\n" + "-"*40 + "\n")

=== Raw Distribution for: content_updated_days ===
content_updated_days
(1.999, 11.0]    455604
(11.0, 25.0]     112051
Name: count, dtype: int64
Bin Edges (Thresholds): [ 2. 11. 25.]

=== Raw Distribution for: median_impressions_may ===
median_impressions_may
(-0.001, 20.0]     144645
(20.0, 64.5]       139470
(64.5, 224.5]      141709
(224.5, 4963.5]    141831
Name: count, dtype: int64
Bin Edges (Thresholds): [   0.    20.    64.5  224.5 4963.5]

=== Raw Distribution for: median_clicks_may ===
median_clicks_may
(-0.001, 37.0]    567655
Name: count, dtype: int64
Bin Edges (Thresholds): [ 0. 37.]

=== Raw Distribution for: median_engaged_sessions_may ===
median_engaged_sessions_may
(-0.001, 3.0]    567655
Name: count, dtype: int64
Bin Edges (Thresholds): [0. 3.]

=== Raw Distribution for: median_avg_position_may ===
median_avg_position_may
(-0.001, 7.057]      140211
(7.057, 11.091]      140220
(11.091, 24.458]     140212
(24.458, 105.167]    140201
Name: count, dtype: int64
Bin Edges 

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

<b>Higher Positions and higher impressions grow linearly both in long term(30d) and short term(daily).</b>

In [13]:
import numpy as np

conditions_pos = [
    feature_frame_may["avg_position_prev30"] <= 10,
    (feature_frame_may["avg_position_prev30"] > 10)
    & (feature_frame_may["avg_position_prev30"] <= 20),
    (feature_frame_may["avg_position_prev30"] >= 20)
    & (feature_frame_may["avg_position_prev30"] <= 30),    
    feature_frame_may["avg_position_prev30"] > 30,
]
choices_pos = ["Pos 1-10", "Pos 10-20", "Pos 20-30", "Pos > 30"]

feature_frame_may["position_bucket"] = np.select(
    conditions_pos, choices_pos, default="Other"
)

# 2. Build Bucket Table for Signal 2
signal_1_bucket = (
    feature_frame_may.groupby("position_bucket")
    .agg(
        N=("content_hash_id", "count"),
        median_imp_30d_avg=("imp_prev30_avg", "median"),
        median_clicks_30d_avg=("clicks_prev30_avg", "median"),
        median_engaged_sessions_30d_avg=("engaged_sessions_prev30_avg", "median"),
        mean_median_clicks=("median_clicks_may", "mean"),
        mean_median_impressions=("median_impressions_may", "mean"),
        mean_median_engaged_sessions=("median_engaged_sessions_may", "mean"),
        mean_median_sum_position=("median_avg_position_may","median")
    )
    .reset_index()
)

print("\n--- SIGNAL 1 BUCKET TABLE: QUICK-WIN POSITION ZONE ---")
print(signal_1_bucket)


--- SIGNAL 1 BUCKET TABLE: QUICK-WIN POSITION ZONE ---
  position_bucket       N  median_imp_30d_avg  median_clicks_30d_avg  \
0        Pos 1-10  225709            0.333333                    0.0   
1       Pos 10-20   97593            0.200000                    0.0   
2       Pos 20-30   54766            0.133333                    0.0   
3        Pos > 30  189587            0.166667                    0.0   

   median_engaged_sessions_30d_avg  mean_median_clicks  \
0                         0.033333            1.566634   
1                         0.000000            0.656374   
2                         0.000000            0.285478   
3                         0.000000            0.112096   

   mean_median_impressions  mean_median_engaged_sessions  \
0               405.531631                      0.021313   
1               231.578674                      0.007255   
2               127.288847                      0.001853   
3                83.481470                      0.00

**Mixed:** Daily metrics of mean_median_impressions confirm the signal but 30d aggregated metrics of median_imp_30d_avg don't.
Position>30 disrupts the linear trend. 

**Peak performance of each content could be tracked almost instantly.**

In [102]:
# 1. Define conditions and choices for content age buckets
conditions_age = [
    feature_frame_may["content_updated_days"] <= 7,
    (feature_frame_may["content_updated_days"] > 7) 
    & (feature_frame_may["content_updated_days"] <= 14),
    (feature_frame_may["content_updated_days"] > 14) 
    & (feature_frame_may["content_updated_days"] <= 21),
    (feature_frame_may["content_updated_days"] > 21) 
    & (feature_frame_may["content_updated_days"] <= 28),
    feature_frame_may["content_updated_days"] > 28,
]

choices_age = ["0-7 days", "7-14 days", "14-21 days", "21-28 days", "> 28 days"]

feature_frame_may["content_age_bucket"] = np.select(
    conditions_age, choices_age, default="Unknown"
)

# 2. Build Bucket Table for Content Age (Peak Performance Tracking)
content_age_bucket_table = (
    feature_frame_may.groupby("content_age_bucket")
    .agg(
        N=("content_hash_id", "count"),
        median_imp_30d_avg=("imp_prev30_avg", "median"),
        median_clicks_30d_avg=("clicks_prev30_avg", "median"),
        median_engaged_sessions_30d_avg=("engaged_sessions_prev30_avg", "median"),
        mean_median_clicks=("median_clicks_may", "mean"),
        mean_median_impressions=("median_impressions_may", "mean"),
        mean_median_engaged_sessions=("median_engaged_sessions_may", "mean"),
        mean_median_sum_position=("median_avg_position_may", "mean"),
        median_positions_30d_avg=("avg_position_prev30","median")
    )
    .reset_index()
)

print("\n--- CONTENT AGE BUCKET TABLE: PEAK PERFORMANCE ZONE ---")
print(content_age_bucket_table)


--- CONTENT AGE BUCKET TABLE: PEAK PERFORMANCE ZONE ---
  content_age_bucket       N  median_imp_30d_avg  median_clicks_30d_avg  \
0           0-7 days   38831            0.166667                    0.0   
1         14-21 days   26974            0.233333                    0.0   
2         21-28 days     204            0.500000                    0.0   
3          7-14 days  501646            0.233333                    0.0   

   median_engaged_sessions_30d_avg  mean_median_clicks  \
0                         0.000000            0.380495   
1                         0.033333            0.718896   
2                         0.033333            4.470588   
3                         0.000000            0.836185   

   mean_median_impressions  mean_median_engaged_sessions  \
0               136.577992                      0.006812   
1               269.806443                      0.007878   
2               989.343137                      0.000000   
3               247.480602          

**FALSE**  The 0-7 days do not show the peak. 

Though we have 21-28 days as the winner here but still we have less number of data for that category compared to toehr categories to draw a conclusion. So, we'll chose the second winner here i.e. 7-14 days.

<b>Navigational has the best performance compared to other search intents.</b>

In [42]:
# 1. Build Bucket Table grouped by main_intent with all metrics
intent_performance_table = (
    feature_frame_may.groupby("main_intent")
    .agg(
        N=("content_hash_id", "count"),
        median_imp_30d_avg=("imp_prev30_avg", "median"),
        median_clicks_30d_avg=("clicks_prev30_avg", "median"),
        median_engaged_sessions_30d_avg=("engaged_sessions_prev30_avg", "median"),
        mean_median_clicks=("median_clicks_may", "mean"),
        mean_median_impressions=("median_impressions_may", "mean"),
        mean_median_engaged_sessions=("median_engaged_sessions_may", "mean"),
        mean_median_avg_position=("median_avg_position_may", "median"),
    )
    .reset_index()
)

# 2. Calculate percentage share of total content for each intent category
total_content_count = intent_performance_table["N"].sum()
intent_performance_table["pct_share"] = (
    intent_performance_table["N"] / total_content_count
) * 100

# Sort by volume (N) descending
intent_performance_table = intent_performance_table.sort_values(
    by="N", ascending=False
).reset_index(drop=True)

intent_performance_table[["main_intent"]] = encoder.inverse_transform(
    intent_performance_table[["main_intent"]]
)
# 3. View all columns
print("\n--- CONTENT PERFORMANCE TABLE BY MAIN INTENT ---")
# Set pandas options to display all columns clearly if needed
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

intent_performance_table


--- CONTENT PERFORMANCE TABLE BY MAIN INTENT ---


,main_intent,N,median_imp_30d_avg,median_clicks_30d_avg,median_engaged_sessions_30d_avg,mean_median_clicks,mean_median_impressions,mean_median_engaged_sessions,mean_median_avg_position,pct_share
0,informational,383821,0.200000,0.0,0.0,0.878628,260.437242,0.006810,10.867647,67.615189
1,transactional,98606,0.233333,0.0,0.0,0.606951,212.972679,0.023371,10.910417,17.370762
2,commercial,83293,0.233333,0.0,0.0,0.662493,188.542375,0.010331,12.101449,14.673173
3,navigational,1935,0.233333,0.0,0.0,1.179070,136.833333,0.000000,8.108040,0.340876


In [50]:
import numpy as np
import pandas as pd

# Set display formatting for clarity
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# -------------------------------------------------------------------------
# Step 1: Compute Global Baselines from Data (To avoid hardcoded thresholds)
# -------------------------------------------------------------------------
global_imp_mean = feature_frame_may["median_impressions_may"].mean()
global_clicks_mean = feature_frame_may["median_clicks_may"].mean()
global_eng_mean = feature_frame_may["median_engaged_sessions_may"].mean()

# Historical baselines (Prev 30 Days)
hist_imp_med = feature_frame_may["imp_prev30_avg"].median()
hist_clicks_med = feature_frame_may["clicks_prev30_avg"].median()

# -------------------------------------------------------------------------
# Step 2: Build Main Intent Signal Bucket Table
# -------------------------------------------------------------------------
intent_performance_table = (
    feature_frame_may.groupby("main_intent")
    .agg(
        n=("content_hash_id", "count"),
        median_imp_30d_avg=("imp_prev30_avg", "median"),
        median_clicks_30d_avg=("clicks_prev30_avg", "median"),
        median_engaged_sessions_30d_avg=("engaged_sessions_prev30_avg", "median"),
        mean_median_clicks=("median_clicks_may", "mean"),
        mean_median_impressions=("median_impressions_may", "mean"),
        mean_median_engaged_sessions=("median_engaged_sessions_may", "mean"),
        mean_median_sum_position=("median_avg_position_may", "median"),
    )
    .reset_index()
)

# Reverse transform encoder for human-readable intent names
intent_performance_table[["main_intent"]] = encoder.inverse_transform(
    intent_performance_table[["main_intent"]]
)

# Add share percentage
total_content_count = intent_performance_table["n"].sum()
intent_performance_table["pct_share"] = (
    intent_performance_table["n"] / total_content_count
) * 100

# -------------------------------------------------------------------------
# Step 3: Evaluate Signal Logic & Assign Verdicts per Intent
# -------------------------------------------------------------------------
verdicts = []
actions = []
reason_codes = []

for idx, row in intent_performance_table.iterrows():
    # Signal: High Clicks + Low Engaged Sessions -> Landing Page Friction
    if (row["mean_median_clicks"] >= global_clicks_mean) and (
        row["mean_median_engaged_sessions"] < global_eng_mean
    ):
        verdicts.append("CONFIRMED")
        actions.append("Content Fix")
        reason_codes.append("CLICK_ENGAGEMENT_DISCONNECT")

    # Signal: Page 2 Position + High Impressions -> Striking Distance Opportunity
    elif (10 < row["mean_median_sum_position"] <= 20) and (
        row["mean_median_impressions"] >= global_imp_mean
    ):
        verdicts.append("CONFIRMED")
        actions.append("Push to Higher Positions")
        reason_codes.append("PAGE2_STRIKING_DISTANCE")

    # Signal: High Impressions + Low Clicks -> Snippet Mismatch
    elif (row["mean_median_sum_position"] <= 10) and (
        row["mean_median_clicks"] < global_clicks_mean
    ):
        verdicts.append("CONFIRMED")
        actions.append("Snippet Fix")
        reason_codes.append("HIGH_IMP_LOW_CTR_TOP10")

    else:
        verdicts.append("MIXED")
        actions.append("No Action")
        reason_codes.append("BASELINE_PERFORMING")

intent_performance_table["verdict"] = verdicts
intent_performance_table["suggested_action"] = actions
intent_performance_table["reason_code"] = reason_codes

# Sort by count descending
intent_performance_table = intent_performance_table.sort_values(
    by="n", ascending=False
).reset_index(drop=True)

# -------------------------------------------------------------------------
# Step 4: Display Output for Notebook Section 1
# -------------------------------------------------------------------------
print("\n=== TASK 1: MAIN INTENT SIGNAL BUCKET TABLE ===")
print(f"Total Sample Size Printed (n): {total_content_count}\n")
intent_performance_table[
        [
            "main_intent",
            "n",
            "pct_share",
            "mean_median_sum_position",
            "mean_median_impressions",
            "mean_median_clicks",
            "mean_median_engaged_sessions",
            "verdict",
            "suggested_action",
            "reason_code",
        ]
    ]


=== TASK 1: MAIN INTENT SIGNAL BUCKET TABLE ===
Total Sample Size Printed (n): 567655



,main_intent,n,pct_share,mean_median_sum_position,mean_median_impressions,mean_median_clicks,mean_median_engaged_sessions,verdict,suggested_action,reason_code
0,informational,383821,67.615189,10.867647,260.437242,0.878628,0.006810,CONFIRMED,Content Fix,CLICK_ENGAGEMENT_DISCONNECT
1,transactional,98606,17.370762,10.910417,212.972679,0.606951,0.023371,MIXED,No Action,BASELINE_PERFORMING
2,commercial,83293,14.673173,12.101449,188.542375,0.662493,0.010331,MIXED,No Action,BASELINE_PERFORMING
3,navigational,1935,0.340876,8.108040,136.833333,1.179070,0.000000,CONFIRMED,Content Fix,CLICK_ENGAGEMENT_DISCONNECT


**FALSE**: navigational was comparatively the least perfoming category here.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Signal chosen: First, Flag chosen: snippet_fix**
1. Does the Data Support the Assumption?
Yes, but only when using daily averages.

The Assumption: Pages with better search position buckets (closer to Pos 1) receive higher search impressions and engagement because search engines find their content more relevant.

Daily Metric Proof (mean_median_impressions):

Pos 1–10: 405.5

Pos 10–20: 231.6

Pos 20–30: 127.3

Pos > 30: 83.5

Verdict: This monotonic step-down confirms the rule. Better position strongly correlates with higher daily impression volume.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

the content needs to be fixed. 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.